# 06 — Coupling-graph routing

Yen *k*-shortest paths from **ligand** contacts to positions among the lowest
deconvolution scores (default: ESM-2 650M vs −SPURS).

The graph is an EVcouplings `*CouplingScoresCompared_all.csv`
(shipped with the example; for a new protein download it from
[evcouplings.org](https://evcouplings.org)).

## Load previous outputs

In [1]:
from IPython.display import display
import pandas as pd

from dualde import load_example
from dualde.contacts import attach_minimum_distance
from dualde.routing import run_information_transfer_pipeline

EXAMPLE, cfg = load_example()
OUT = EXAMPLE / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
LIGAND_NAMES = list(cfg["ligand_names"])

deconv = pd.read_csv(OUT / "deconvolution.csv")
contacts = pd.read_csv(OUT / "contacts.csv")
func = pd.read_csv(OUT / "functional_residues.csv")
coup_files = sorted((EXAMPLE / "couplings").glob("*CouplingScoresCompared_all.csv"))
if not coup_files:
    raise FileNotFoundError(f"No *CouplingScoresCompared_all.csv under {EXAMPLE / 'couplings'}")
coup_path = coup_files[0]
coup = pd.read_csv(coup_path)

print("deconv", deconv.shape, "contacts", contacts.shape, "func", func.shape)
print("couplings", coup.shape, "from", coup_path.name)

deconv (2844, 29) contacts (9163, 9) func (52, 5)
couplings (7503, 15) from b0.5_CouplingScoresCompared_all.csv


Coupling table from the EVcouplings webserver (needs `i`, `j`, `probability`, `dist`):

In [2]:
print("coupling columns:", list(coup.columns))
display(coup[["i", "j", "probability", "dist"]].describe().T)
display(coup.head())

coupling columns: ['i', 'A_i', 'j', 'A_j', 'fn', 'cn', 'segment_i', 'segment_j', 'mad_score', 'probability', 'score', 'dist_intra', 'dist_multimer', 'dist', 'precision']


,count,mean,std,min,25%,50%,75%,max
i,7503.0,54.021858,30.074502,11.000000,30.000000,49.000000,75.000000,140.000000
j,7503.0,97.392776,30.551410,12.000000,75.000000,101.000000,123.000000,141.000000
probability,7503.0,0.240241,0.196305,0.011598,0.113459,0.186082,0.288492,1.000000
dist,7503.0,12.356011,5.118126,1.292619,8.600739,12.487660,16.147913,26.200742


,i,A_i,j,A_j,fn,cn,segment_i,segment_j,mad_score,probability,score,dist_intra,dist_multimer,dist,precision
0,65,W,71,H,0,1.610785,A,A,26.163987,1.000000,16.721209,3.161111,14.659354,3.161111,1.0
1,119,S,121,E,0,1.375301,A,A,22.381647,0.999999,14.178140,2.504789,25.820062,2.504789,1.0
2,22,S,28,C,0,1.210362,A,A,19.732400,0.999996,12.403793,4.398179,12.570272,4.398179,1.0
3,110,K,112,V,0,1.207799,A,A,19.691234,0.999996,12.351492,2.801678,18.590232,2.801678,1.0
4,50,L,59,C,0,1.150662,A,A,18.773500,0.999991,11.659388,3.630927,15.638187,3.630927,1.0


In [3]:
print("functional residues by type:")
print(func["contact_type"].value_counts())
print("ligand sources (residue_name in", LIGAND_NAMES, "):")
lig = func.loc[func["residue_name"].isin(LIGAND_NAMES)]
display(lig[["target_pos", "residue_name", "minimum_distance", "contact_type"]])

functional residues by type:
contact_type
homodimer    30
ligand       22
Name: count, dtype: int64
ligand sources (residue_name in ['MKB', 'SYW', '4DG', 'UO8', 'RQ2', 'RTW', 'RDZ', '71V'] ):


,target_pos,residue_name,minimum_distance,contact_type
6,15,MKB,3.241971,ligand
7,16,SYW,3.119327,ligand
8,34,4DG,2.808717,ligand
9,37,4DG,2.924247,ligand
10,38,SYW,3.204409,ligand
11,44,UO8,3.203179,ligand
12,45,RQ2,2.672945,ligand
13,46,RTW,3.419643,ligand
14,47,RTW,2.858231,ligand
15,48,SYW,2.930096,ligand


## Run routing

In [4]:
scores = attach_minimum_distance(deconv, contacts)

score_col = cfg["score_col"]
if score_col not in scores.columns:
    raise ValueError(f"Run notebook 05 first (need {score_col}).")
print("score_col:", score_col)

result = run_information_transfer_pipeline(
    all_merged_df=scores,
    couplings_file=str(coup_path),
    functional_residues_file=str(OUT / "functional_residues.csv"),
    active_site_ligands=LIGAND_NAMES,
    activity_wt=cfg["activity_wt"],
    abundance_wt=cfg["abundance_wt"],
    score_col=score_col,
    low_score_percentile=cfg["low_score_percentile"],
    prob_threshold=cfg["prob_threshold"],
    n_paths=cfg["n_paths"],
    weighted=True,
    dist_cutoff=cfg["dist_cutoff"],
    functional_dist_cutoff=cfg["functional_dist_cutoff"],
    seq_len=cfg["seq_len"],
    verbose=True,
)

paths = result["paths_df"]
pairs = result["pairs_df"]
paths.to_csv(OUT / "coupling_paths.csv", index=False)
pairs.to_csv(OUT / "coupling_pairs.csv", index=False)
print("wrote", OUT / "coupling_paths.csv")

score_col: ESM2_650M_neg_spurs_model


PIPELINE CHECKS

[1] Scores / cutoffs
  n mutants:              2844
  activity_wt:            0.45
  abundance_wt:           0.45
  n SBI mutants:          236
  n functional_site True: 882
  target_pos range:       2–164

[2] Couplings
  couplings_file: /lustre/scratch125/gengen/projects_v2/sgb_enzyme_design/evedesign_prelim/dualde/examples/NUDT15_HUMAN/couplings/b0.5_CouplingScoresCompared_all.csv
  dist_cutoff: 5.0
  seq_len: 164
  n_raw_positive_prob: 7503
  n_positive_prob_after_dist_filter: 706
  max_ij_in_file: 141
  grid_shape: (164, 164)
  n_nonzero_grid: 706

[3] Endpoints
  score_col:              ESM2_650M_neg_spurs_model
  low_score_percentile:   1.0% -> n_take=28
  ligands requested:      ['4DG', '71V', 'MKB', 'RDZ', 'RQ2', 'RTW', 'SYW', 'UO8']
  ligands in file:        ['4DG', '71V', 'MKB', 'RDZ', 'RQ2', 'RTW', 'SYW', 'UO8']
  active-site SOURCES:    22  [15, 16, 34, 37, 38, 44, 45, 46, 47, 48, 49, 67, 83, 88, 90, 92, 94, 116, 135, 136, 137, 138]
  bottom mutants:      

## Inspect paths

In [5]:
print("n paths:", len(paths), "n pairs:", len(pairs))
print("pairs with a path:", int((pairs["n_paths_found"] > 0).sum()) if len(pairs) else 0)
print("sources:", sorted(paths["source"].unique()) if len(paths) else [])
print("targets:", sorted(paths["target"].unique()) if len(paths) else [])
display(paths.head(10))

n paths: 372 n pairs: 130
pairs with a path: 76
sources: [np.int64(15), np.int64(16), np.int64(37), np.int64(38), np.int64(44), np.int64(45), np.int64(46), np.int64(47), np.int64(48), np.int64(49), np.int64(83), np.int64(88), np.int64(90), np.int64(92), np.int64(94), np.int64(116), np.int64(135), np.int64(136), np.int64(137), np.int64(138)]
targets: [np.int64(15), np.int64(51), np.int64(66), np.int64(82), np.int64(113)]


,source,target,path_rank,path,path_str,n_edges,path_cost,path_coupling_score,min_edge_prob,mean_edge_prob,edge_probs
0,15,51,1,"[15, 17, 18, 60, 97, 56, 82, 91, 92, 49, 51]",15 -> 17 -> 18 -> 60 -> 97 -> 56 -> 82 -> 91 -...,10,0.158494,-0.158494,0.956867,0.984151,"[0.9932280867167422, 0.9752958122175348, 0.999..."
1,15,51,2,"[15, 17, 96, 94, 92, 49, 51]",15 -> 17 -> 96 -> 94 -> 92 -> 49 -> 51,6,0.160097,-0.160097,0.914947,0.973317,"[0.9932280867167422, 0.9653329244451156, 0.914..."
2,15,51,3,"[15, 17, 18, 60, 97, 77, 80, 95, 56, 82, 91, 9...",15 -> 17 -> 18 -> 60 -> 97 -> 77 -> 80 -> 95 -...,13,0.167856,-0.167856,0.956867,0.987088,"[0.9932280867167422, 0.9752958122175348, 0.999..."
3,15,51,4,"[15, 17, 18, 60, 97, 95, 56, 82, 91, 92, 49, 51]",15 -> 17 -> 18 -> 60 -> 97 -> 95 -> 56 -> 82 -...,11,0.176947,-0.176947,0.956867,0.983914,"[0.9932280867167422, 0.9752958122175348, 0.999..."
4,15,51,5,"[15, 17, 18, 60, 97, 77, 80, 56, 82, 91, 92, 4...",15 -> 17 -> 18 -> 60 -> 97 -> 77 -> 80 -> 56 -...,12,0.181297,-0.181297,0.956867,0.984892,"[0.9932280867167422, 0.9752958122175348, 0.999..."
5,15,82,1,"[15, 17, 18, 60, 97, 56, 82]",15 -> 17 -> 18 -> 60 -> 97 -> 56 -> 82,6,0.082038,-0.082038,0.959076,0.986327,"[0.9932280867167422, 0.9752958122175348, 0.999..."
6,15,82,2,"[15, 17, 18, 60, 97, 77, 80, 95, 56, 82]",15 -> 17 -> 18 -> 60 -> 97 -> 77 -> 80 -> 95 -...,9,0.091401,-0.091401,0.959076,0.989844,"[0.9932280867167422, 0.9752958122175348, 0.999..."
7,15,82,3,"[15, 17, 18, 60, 97, 95, 56, 82]",15 -> 17 -> 18 -> 60 -> 97 -> 95 -> 56 -> 82,7,0.100491,-0.100491,0.959076,0.985644,"[0.9932280867167422, 0.9752958122175348, 0.999..."
8,15,82,4,"[15, 17, 18, 60, 97, 77, 80, 56, 82]",15 -> 17 -> 18 -> 60 -> 97 -> 77 -> 80 -> 56 -...,8,0.104842,-0.104842,0.959076,0.986895,"[0.9932280867167422, 0.9752958122175348, 0.999..."
9,15,82,5,"[15, 17, 18, 60, 97, 77, 95, 56, 82]",15 -> 17 -> 18 -> 60 -> 97 -> 77 -> 95 -> 56 -...,8,0.107869,-0.107869,0.959076,0.986516,"[0.9932280867167422, 0.9752958122175348, 0.999..."


In [6]:
if len(paths):
    print("shortest paths (fewest edges):")
    display(paths.nsmallest(8, "n_edges")[["source", "target", "path_rank", "n_edges", "min_edge_prob", "path_str"]])
    print("pair status (first 12):")
    display(pairs.head(12))
else:
    print("No paths — loosen prob_threshold or check that notebook 01 found ligand contacts.")

shortest paths (fewest edges):


,source,target,path_rank,n_edges,min_edge_prob,path_str
161,49,51,1,1,0.977910,49 -> 51
186,83,82,5,1,0.861923,83 -> 82
287,116,113,1,1,0.998542,116 -> 113
25,16,82,1,2,0.939106,16 -> 56 -> 82
95,45,15,1,2,0.900980,45 -> 17 -> 15
135,47,15,1,2,0.933891,47 -> 17 -> 15
155,48,66,1,2,0.737568,48 -> 63 -> 66
183,83,82,2,2,0.941139,83 -> 81 -> 82


pair status (first 12):


,source,target,n_paths_found,status
0,15,51,5,ok
1,15,66,0,no path
2,15,67,0,no path
3,15,82,5,ok
4,15,113,5,ok
5,16,15,5,ok
6,16,51,5,ok
7,16,66,0,no path
8,16,67,0,no path
9,16,82,5,ok
